## Feature Engineering

### SETUP

In [30]:
! pip install pyod

In [31]:
import pandas as pd
from sklearn.preprocessing import Normalizer
from pyod.models.knn import KNN

In [46]:
columns = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
df = pd.read_csv('../data/housing.csv', header=None, sep=r"\s+", names=columns, nrows = None)

In [47]:
df

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273.0,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273.0,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273.0,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273.0,21.0,393.45,6.48,22.0


## Tratando dados faltando ou duplicatas

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    int64  
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    int64  
 9   TAX      506 non-null    float64
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
 13  MEDV     506 non-null    float64
dtypes: float64(12), int64(2)
memory usage: 55.5 KB


Não há a existência de dados faltantes no conjunto de dados. Os tipos de informações parecem ter sentido também.

In [49]:
df = df.drop("B", axis=1)

Como mencionado no notebook de análise exploratória, é muito importante considerar questoões de conformidade e ética ao treinar o modelo. Isso porque existe risco eq organização sofrer danos reputacionais caso o modelo tome decisões baseadas em informações sensíveis, como raça ou religião dos individuso. Por isso, é crucial tomar medidas preventivas, como a remoção de colunas que possam conter esse tipo de dado.

A coluna removida pode conter informações informações potencialmente sensíveis, que se utilizada pelo modelo, pode levar a decisões enviesadas ou discriminatórias.

In [52]:
df.drop_duplicates()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273.0,21.0,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273.0,21.0,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273.0,21.0,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273.0,21.0,6.48,22.0


## Detecção e tratamento de Outliers

In [53]:
from pyod.models.knn import KNN

detector = KNN()
detector.fit(df)

prev = detector.labels_

In [54]:
outliers = []
for i in range(len(prev)):
  if prev[i] == 1:
    outliers.append(i)

In [55]:
df.drop(outliers, axis='rows', inplace=True)

In [56]:
df.shape

(455, 13)

Removemos os outliers pois, a quantidade de registros não era exatamente grande, e modelos de regressão são muito sensíveis a outliers. Somente para prevenção os registros foram removidos.

## Normalization

In [42]:

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_normalized = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
df_normalized


['scaler.joblib']

A normalização foi escolhida ao invés da padronização dos dados, pois a padronização normalmente se encaixa quando o conjunto de dados tem uma distribuição gaussiana. Voltando a análise exploratória feita anteriormente os dados não possuem essa distribuição em sua maioria. Por isso o motivo da escolha.

Como o único dado categórico que existe no conjunto de dados é a variável CHAS representado por valores booleanos, não existe a necessidade de realizar o encoder deles

In [74]:
df_normalized.to_csv('../data/housing_normalized.csv', index=False)